In [ ]:
import pandas as pd, numpy as np 
import warnings; warnings.filterwarnings('ignore') 
  
df = pd.read_excel('../data/dataset_assurance_ML.xlsx') 
df.to_csv('../data/dataset_assurance_ML.csv', index=False, encoding='utf-8-sig') 
  
df = pd.read_csv('../data/dataset_assurance_ML.csv', encoding='utf-8-sig') 
print(df.shape) 
print(df.isnull().sum())
print(df.duplicated().sum())

(500, 27)
N° Police                 0
Nom                       0
Prénom                    0
Sexe                      0
Âge                       0
Catégorie Prof.           0
Salaire Annuel (€)        0
Ville                     0
Code Postal               0
Type Contrat              0
Statut Contrat            0
Date Souscription         0
Date Échéance             0
Ancienneté (mois)         0
Prime Annuelle (€)        0
Franchise (€)             0
Nb Garanties              0
Type Véhicule             0
Usage Véhicule            0
Puissance Fiscale (CV)    0
Valeur Véhicule (€)       0
Coeff. Bonus-Malus        0
Nb Sinistres (3 ans)      0
Montant Sinistres (€)     0
Dernier Sinistre          0
Score Risque (0-100)      0
Résiliation               0
dtype: int64
0


In [8]:
TARGET = 'Résiliation' 
print(df[TARGET].value_counts()) 
print(df[TARGET].value_counts(normalize=False).round(2)) 

Résiliation
0    450
1     50
Name: count, dtype: int64
Résiliation
0    450
1     50
Name: count, dtype: int64


Non. Une classe pèse 9 fois plus que l'autre, on parle de classes déséquilibrées.

In [9]:
print(pd.crosstab(df['Statut Contrat'], df[TARGET])) 

Résiliation       0   1
Statut Contrat         
Actif           450   0
Résilié           0  36
Suspendu          0  14


Non. Le statut est une conséquence de la résiliation pas une cause.

In [11]:
num_cols = ['Âge', 'Salaire Annuel (€)', 'Prime Annuelle (€)', 'Ancienneté (mois)', 
            'Coeff. Bonus-Malus', 'Nb Sinistres (3 ans)', 
            'Montant Sinistres (€)', 'Score Risque (0-100)']
            
cat_cols = ['Type Contrat', 'Catégorie Prof.', 'Usage Véhicule', 'Dernier Sinistre'] 
X = df[num_cols + cat_cols] 
y = df[TARGET] 
print(X.shape, y.shape)           

(500, 12) (500,)


In [12]:
print(X[num_cols].corrwith(y).round(3).sort_values(ascending=False)) 
print(df.groupby('Dernier Sinistre')[TARGET].mean().round(2).sort_values())

Nb Sinistres (3 ans)     0.455
Score Risque (0-100)     0.441
Coeff. Bonus-Malus       0.412
Montant Sinistres (€)    0.284
Prime Annuelle (€)       0.041
Âge                     -0.006
Salaire Annuel (€)      -0.012
Ancienneté (mois)       -0.055
dtype: float64
Dernier Sinistre
Aucun                    0.03
Incendie                 0.16
Catastrophe naturelle    0.18
Accident                 0.26
Bris de glace            0.30
Vol                      0.30
Dégât des eaux           0.36
Name: Résiliation, dtype: float64


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(2), y_test.mean().round(2))

(400, 12) (100, 12)
0.1 0.1


In [15]:
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import StandardScaler, OneHotEncoder 
  
preprocessor = ColumnTransformer([ 
    ('num', StandardScaler(), num_cols), 
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols), 
]) 

In [16]:
from sklearn.pipeline import Pipeline 
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier 
  
candidats = { 
    'Régression Logistique': LogisticRegression( 
        max_iter=1000, class_weight='balanced', random_state=42), 
    'Random Forest': RandomForestClassifier( 
        n_estimators=300, max_depth=4, min_samples_leaf=10, 
        class_weight='balanced', random_state=42), 
} 
pipelines = {nom: Pipeline([('prep', preprocessor), ('model', algo)]) 
            for nom, algo in candidats.items()}

In [19]:
from sklearn.model_selection import cross_val_score

for nom, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    print(f'{nom:22s} AUC = {scores.mean():.3f} ± {scores.std():.3f}')

Régression Logistique  AUC = 0.745 ± 0.113
Random Forest          AUC = 0.806 ± 0.096


In [25]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)

pipeline = pipelines['Random Forest']
pipeline.fit(X_train, y_train)
y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print('Accuracy :', round(accuracy_score(y_test, y_pred), 3))
print('F1       :', round(f1_score(y_test, y_pred), 3))
print('ROC-AUC  :', round(roc_auc_score(y_test, y_proba), 3))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Reste', 'Résilie']))

Accuracy : 0.82
F1       : 0.438
ROC-AUC  : 0.861
[[75 15]
 [ 3  7]]
              precision    recall  f1-score   support

       Reste       0.96      0.83      0.89        90
     Résilie       0.32      0.70      0.44        10

    accuracy                           0.82       100
   macro avg       0.64      0.77      0.67       100
weighted avg       0.90      0.82      0.85       100



**Q1.** Non. Le modèle « toujours reste » a 90 % d'accuracy mais ne détecte
aucune résiliation (recall = 0 %). Le nôtre a 79 % d'accuracy mais détecte
5 résiliations sur 10 (recall = 50 %). L'accuracy est trompeuse sur des
classes déséquilibrées : le nôtre est plus utile.

**Q2.** Le faux négatif coûte le plus cher : un client qui part sans qu'on
ait réagi est un contrat perdu. Un faux positif ne coûte qu'un appel inutile.

**Q3.** Il faut baisser le seuil sous 0,5. Le modèle signale ainsi plus de
clients à risque : on détecte plus de départs (moins de faux négatifs), au
prix de plus de fausses alertes, ce qui reste acceptable.

In [26]:
import joblib, os

os.makedirs('models', exist_ok=True)
joblib.dump(pipeline, 'models/pipeline_resiliation.pkl')
print(os.path.getsize('models/pipeline_resiliation.pkl') / 1024, 'Ko')

486.845703125 Ko


In [27]:
import json

meta = {
    'modele': 'Random Forest',
    'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3),
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'num_ranges': {c: {'min': float(X[c].min()),
                       'max': float(X[c].max()),
                       'median': float(X[c].median())} for c in num_cols},
    'cat_values': {c: sorted(X[c].unique().tolist()) for c in cat_cols},
}

with open('models/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

In [28]:
import pandas as pd
import joblib

modele = joblib.load('models/pipeline_resiliation.pkl')

client = pd.DataFrame([{
    'Âge': 34,
    'Salaire Annuel (€)': 28000,
    'Prime Annuelle (€)': 950,
    'Ancienneté (mois)': 6,
    'Coeff. Bonus-Malus': 1.25,
    'Nb Sinistres (3 ans)': 3,
    'Montant Sinistres (€)': 4200,
    'Score Risque (0-100)': 72,
    'Type Contrat': 'Bronze',
    'Catégorie Prof.': 'Entrepreneur',
    'Usage Véhicule': 'Professionnel',
    'Dernier Sinistre': 'Vol',
}])

print('Classe :', modele.predict(client))
print('Proba :', modele.predict_proba(client)[0, 1].round(3))

Classe : [1]
Proba : 0.828


In [29]:
client_fidele = client.copy()
client_fidele['Nb Sinistres (3 ans)'] = 0
client_fidele['Montant Sinistres (€)'] = 0
client_fidele['Ancienneté (mois)'] = 200
client_fidele['Type Contrat'] = 'Gold'
client_fidele['Dernier Sinistre'] = 'Aucun'

print('Classe :', modele.predict(client_fidele))
print('Proba :', modele.predict_proba(client_fidele)[0, 1].round(3))

Classe : [1]
Proba : 0.525


In [30]:
try:
    modele.predict(client.drop(columns=['Score Risque (0-100)']))
except Exception as e:
    print('ERREUR :', e)

ERREUR : columns are missing: {'Score Risque (0-100)'}


In [ ]:
import os
import json
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Chargement : recopie ici la ligne de ton étape 1
df = pd.read_csv('../data/dataset_assurance_ML.csv')

# 2. Variables
TARGET = 'Résiliation'
num_cols = ['Âge', 'Salaire Annuel (€)', 'Prime Annuelle (€)', 'Ancienneté (mois)',
            'Coeff. Bonus-Malus', 'Nb Sinistres (3 ans)', 'Montant Sinistres (€)',
            'Score Risque (0-100)']
cat_cols = ['Type Contrat', 'Catégorie Prof.', 'Usage Véhicule', 'Dernier Sinistre']
X = df[num_cols + cat_cols]
y = df[TARGET]

# 3. Séparation train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Pipeline Random Forest
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=10,
        class_weight='balanced', random_state=42)),
])

# 5. Entraînement et AUC sur le test
pipeline.fit(X_train, y_train)
y_proba = pipeline.predict_proba(X_test)[:, 1]

# 6. Sauvegardes
os.makedirs('models', exist_ok=True)
joblib.dump(pipeline, 'models/pipeline_resiliation.pkl')

meta = {
    'modele': 'Random Forest',
    'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3),
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'num_ranges': {c: {'min': float(X[c].min()),
                       'max': float(X[c].max()),
                       'median': float(X[c].median())} for c in num_cols},
    'cat_values': {c: sorted(X[c].unique().tolist()) for c in cat_cols},
}
with open('models/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('Modèle et métadonnées sauvegardés. AUC test =', meta['auc_test'])